# Notebook 8.2  Who is speaking, and whether cleaning the audio helps

*Equal Error Rate, a Diarization Error Rate in three parts, and an enhancement test judged by the transcript.*

---

*Companion notebook to* **Introduction to Arabic Speech Technology** *by Hend S. Al-Khalifa.*

**How to run.** Open this notebook in Google Colab or run it locally with the
pinned environment in `requirements.txt`. Every notebook in this series runs end
to end with **no downloads and no accounts**: where a real corpus or a
pretrained model is unavailable, a clearly marked fallback stands in for it, and
the notebook says which path it took. Cells that need a download are marked
`OPTIONAL` and are safe to skip.

**On data.** Where you substitute a real corpus, record its release version and
its licence in the provenance cell at the end. A result without them is not
reproducible, which is the habit this book asks for in every chapter.

<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/notebooks-archive/ch08_speaker_and_noise.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## What this notebook does

Three measurements that the chapter argues are usually reported wrongly.

1. **Speaker verification.** Build genuine and impostor trials from
   speaker-disjoint embeddings, sweep the threshold, and find the Equal Error
   Rate. Then look at what happens away from the equal error point, because no
   deployed system runs there.
2. **Diarization.** Cluster segment embeddings, then compute the Diarization
   Error Rate in its three parts: false alarm, missed speech, and speaker
   confusion. One number hides which of the three is your problem.
3. **Enhancement.** Mix noise into speech, denoise it, and measure the result
   two ways: with a signal-quality score and with the transcript. Section 8.7's
   uncomfortable finding is that these two can disagree, and this is where you
   can watch them disagree.

Synthetic fallbacks throughout, so it runs with no downloads.

In [ ]:
NOTEBOOK = 'ch08_speaker_and_noise.ipynb'

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)
SR = 16000
print('ready')

## 1. Speaker verification and the Equal Error Rate

A verification system compares two utterances and returns a score. Sweep a
threshold across those scores and two errors trade off: the false acceptance
rate falls as the threshold rises, and the false rejection rate climbs. The
Equal Error Rate is the point where they cross.

It is a convenient single number and it describes an operating point almost
nobody uses. A bank does not accept a one in twenty impostor rate; it moves the
threshold until false acceptance is negligible and lives with the rejections.

In [ ]:
def speaker_embeddings(n_speakers=40, per_speaker=6, dim=128,
                       within=0.12):
    centres = rng.normal(size=(n_speakers, dim))
    centres /= np.linalg.norm(centres, axis=1, keepdims=True)
    X, ids = [], []
    for s in range(n_speakers):
        noise = rng.normal(scale=within, size=(per_speaker, dim))
        v = centres[s] + noise
        X.append(v / np.linalg.norm(v, axis=1, keepdims=True))
        ids += [s] * per_speaker
    return np.vstack(X), np.array(ids)


# OPTIONAL: replace with a pretrained speaker encoder (for example ECAPA-TDNN)
# from speechbrain.inference import EncoderClassifier
X, spk = speaker_embeddings()

genuine, impostor = [], []
for i in range(len(X)):
    for j in range(i + 1, len(X)):
        score = float(X[i] @ X[j])
        (genuine if spk[i] == spk[j] else impostor).append(score)
genuine, impostor = np.array(genuine), np.array(impostor)
print(f'{len(genuine)} genuine trials, {len(impostor)} impostor trials')

In [ ]:
thresholds = np.linspace(-1, 1, 601)
far = np.array([(impostor >= t).mean() for t in thresholds])
frr = np.array([(genuine < t).mean() for t in thresholds])
k = int(np.argmin(np.abs(far - frr)))
eer = (far[k] + frr[k]) / 2

plt.figure(figsize=(10, 3.4))
plt.subplot(1, 2, 1)
plt.hist(impostor, bins=60, alpha=0.6, label='impostor', density=True)
plt.hist(genuine, bins=60, alpha=0.6, label='genuine', density=True)
plt.axvline(thresholds[k], color='k', linestyle='--', linewidth=1)
plt.legend()
plt.title('trial scores')
plt.subplot(1, 2, 2)
plt.plot(thresholds, far, label='false acceptance')
plt.plot(thresholds, frr, label='false rejection')
plt.axvline(thresholds[k], color='k', linestyle='--', linewidth=1)
plt.legend()
plt.xlabel('threshold')
plt.title(f'EER = {eer:.1%} at threshold {thresholds[k]:.2f}')
plt.tight_layout()
plt.show()

print(f'EER {eer:.2%}\n')
print('what the same system does at thresholds you might actually deploy:')
for target in (0.01, 0.001):
    idx = int(np.argmin(np.abs(far - target)))
    print(f'  false acceptance {far[idx]:.3%}  ->  '
          f'false rejection {frr[idx]:.1%} at threshold {thresholds[idx]:.2f}')
print('\nThat second column is what the user experiences, and the EER never '
      'mentions it.')

## 2. Diarization, in three parts

Who spoke when. The output is a segmentation with speaker labels, and the error
has three separable components:

- **false alarm**: speech marked where there was none;
- **missed speech**: speech that was not marked;
- **speaker confusion**: speech marked, but attributed to the wrong speaker.

They have different causes and different fixes. A DER of 20 percent that is all
confusion is a clustering problem; the same 20 percent that is all missed
speech is a voice-activity problem. Reporting the total alone tells the reader
neither.

In [ ]:
# a short two-speaker conversation, in seconds
REFERENCE = [(0.0, 3.2, 'A'), (3.2, 5.0, 'B'), (5.0, 8.4, 'A'),
             (8.4, 11.0, 'B'), (11.0, 12.5, 'A')]
HYPOTHESIS = [(0.2, 3.0, 'S1'), (3.0, 5.4, 'S2'), (5.4, 8.0, 'S1'),
              (8.0, 10.4, 'S1'), (10.4, 12.5, 'S2')]


def frames(segments, step=0.01, end=12.5):
    grid = np.arange(0, end, step)
    labels = np.full(len(grid), '', dtype=object)
    for start, stop, who in segments:
        labels[(grid >= start) & (grid < stop)] = who
    return grid, labels


def der(reference, hypothesis, step=0.01):
    _, ref = frames(reference, step)
    _, hyp = frames(hypothesis, step)
    speech = ref != ''
    total = speech.sum()
    false_alarm = ((ref == '') & (hyp != '')).sum()
    missed = (speech & (hyp == '')).sum()
    # map hypothesis labels to reference labels by best overlap
    mapping = {}
    for h in set(hyp) - {''}:
        counts = {r: ((hyp == h) & (ref == r)).sum() for r in set(ref) - {''}}
        mapping[h] = max(counts, key=counts.get)
    mapped = np.array([mapping.get(h, '') for h in hyp], dtype=object)
    confusion = (speech & (hyp != '') & (mapped != ref)).sum()
    return {'false alarm': false_alarm / total, 'missed': missed / total,
            'confusion': confusion / total,
            'DER': (false_alarm + missed + confusion) / total,
            'mapping': mapping}


result = der(REFERENCE, HYPOTHESIS)
for key in ('false alarm', 'missed', 'confusion', 'DER'):
    print(f'{key:>12}: {result[key]:6.1%}')
print(f'\nhypothesis speakers mapped to reference: {result["mapping"]}')
print('The largest of the three components is the one to work on. Here it is '
      'the one the total would never have told you about.')

## 3. Enhancement: the two measurements that disagree

Mix noise into clean speech, denoise it, and score the result twice.

The signal measure below is the scale-invariant signal-to-distortion ratio, one
of the three metrics Section 8.7 names. It is computed against the clean
reference, which is why it can only be measured on simulated mixtures and never
on the field audio a system will actually meet.

The transcript measure is the word error rate of a recognizer reading the
output. Section 8.7 reports a published case where denoising raised SI-SDR and
raised the error rate too. The OPTIONAL cell runs a real recognizer so you can
try to reproduce that on your own audio; without it, this cell shows the signal
side only, and says so.

In [ ]:
def si_sdr(reference, estimate):
    # scale invariant: rescaling the estimate cannot change the score
    reference = reference - reference.mean()
    estimate = estimate - estimate.mean()
    alpha = (estimate @ reference) / (reference @ reference + 1e-12)
    target = alpha * reference
    noise = estimate - target
    return 10 * np.log10((target @ target + 1e-12) / (noise @ noise + 1e-12))


def denoise(noisy, frame=512, hop=128, floor=0.1, noise_frames=20):
    # a Wiener gain: the gain per bin is snr / (1 + snr), with a floor so the
    # gain never reaches zero, because a zeroed bin is where musical noise
    # comes from
    window = np.hanning(frame)
    n_frames = 1 + (len(noisy) - frame) // hop
    stft = np.array([np.fft.rfft(noisy[i * hop:i * hop + frame] * window)
                     for i in range(n_frames)])
    magnitude, phase = np.abs(stft), np.angle(stft)
    noise_power = np.mean(magnitude[:noise_frames] ** 2, axis=0)
    snr = np.maximum(magnitude ** 2 - noise_power, 0) / (noise_power + 1e-12)
    cleaned = magnitude * np.maximum(snr / (1 + snr), floor)
    out = np.zeros(len(noisy))
    norm = np.zeros(len(noisy))
    for i in range(n_frames):
        seg = np.fft.irfft(cleaned[i] * np.exp(1j * phase[i]), n=frame)
        out[i * hop:i * hop + frame] += seg * window
        norm[i * hop:i * hop + frame] += window ** 2
    # the floor on the normalizer matters: at the very first and last samples
    # only one tapered window contributes, and dividing by almost nothing
    # produces a spike that dominates every score computed afterwards
    return out / np.maximum(norm, 1e-3)


t = np.arange(int(2.0 * SR)) / SR
speech = np.sin(2 * np.pi * 180 * t) * (0.5 + 0.5 * np.sin(2 * np.pi * 3 * t))
speech += 0.3 * np.sin(2 * np.pi * 1400 * t) * (t % 0.25 < 0.12)
speech /= np.abs(speech).max()
# a moment of silence at the front, which is where the noise floor is measured
clean = np.concatenate([np.zeros(int(0.4 * SR)), speech])
inside = slice(1000, len(clean) - 1000)

print(f'{"SNR in":>8} {"SI-SDR noisy":>14} {"SI-SDR denoised":>17} '
      f'{"change":>9}')
for snr_db in (20, 10, 5, 0, -5):
    noise = rng.normal(scale=1.0, size=len(clean))
    scale = np.sqrt((clean @ clean) / (noise @ noise) / 10 ** (snr_db / 10))
    noisy = clean + scale * noise
    denoised = denoise(noisy)[:len(clean)]
    a = si_sdr(clean[inside], noisy[inside])
    b = si_sdr(clean[inside], denoised[inside])
    print(f'{snr_db:>6} dB {a:>13.2f} {b:>17.2f} {b - a:>+9.2f}')

print('\nThe signal score improves. Whether the transcript does is a '
      'different question, and the only way to answer it is to run a '
      'recognizer over both and compare the word error rates. That is the '
      'comparison Section 8.7 reports, and the one to reproduce on your own '
      'audio in the cell below.')

In [ ]:
RUN_ASR = False        # needs transformers, torch and real Arabic audio

if RUN_ASR:
    from transformers import pipeline
    import soundfile as sf

    asr = pipeline('automatic-speech-recognition', model='openai/whisper-small',
                   generate_kwargs={'language': 'arabic'})
    CLIPS = []          # [(path, reference_transcript), ...]
    for path, reference in CLIPS:
        wav, sr = sf.read(path)
        noise = rng.normal(scale=0.05, size=len(wav))
        noisy = wav + noise
        denoised = denoise(noisy)[:len(wav)]
        for name, signal in (('clean', wav), ('noisy', noisy),
                             ('denoised', denoised)):
            sf.write(f'/tmp/{name}.wav', signal, sr)
            print(name, asr(f'/tmp/{name}.wav')['text'])
    print('Now compute WER for each condition and compare. If denoising '
          'lowered SI-SDR distortion and raised WER, you have reproduced '
          'the finding.')
else:
    print('skipped: needs a recognizer and real audio')

## Provenance

Fill this in before you quote any number from this notebook. It is the same
information the chapter's Reproducibility Note asks for, and it is the
difference between a result and a screenshot.

In [ ]:
PROVENANCE = {
    'notebook': NOTEBOOK,
    'ran_on': 'fill in the date you ran it',
    'data': 'corpus name and release version, or "synthetic fallback"',
    'licence': 'the licence of the data you used',
    'model': 'model name and revision, or "none"',
    'normalization': 'the normalization applied before scoring',
    'hardware': 'CPU or the GPU model',
}
for k, v in PROVENANCE.items():
    print(f'{k:>15}: {v}')